# ReCaLL (Relative Conditional Log-Likelihood) Membership Inference Attack Recreation

This notebook recreates the **ReCaLL** membership inference attack summarized in `papers/summary/08_recall.md`.

Primary source:

- Roy Xie, Junlin Wang, Ruomin Huang, Minxing Zhang, Rong Ge, Jian Pei, Neil Zhenqiang Gong, Bhuwan Dhingra (Duke University), *ReCaLL: Membership Inference via Relative Conditional Log-Likelihoods*, EMNLP 2024 (main), pp. 8671-8689. arXiv:2406.15968. Project page: royxie.com/recall-project-page. Code: github.com/ruoyuxie/recall.

**Threat model.** ReCaLL is **reference-model-free** and **inference-time only**. The attacker needs only forward passes of the target autoregressive model `M` and a small set of data points *known to be non-members* (obtainable via a knowledge-cutoff date, user-generated text, or synthetic generation). No reference model and no access to the private training distribution are required.

**Core idea.** Build a *fixed* prefix `P = p1 (+) p2 (+) ... (+) pn`, a concatenation of `n` non-member "shots". For a target `x`, compute two log-likelihoods from `M`: the unconditional `LL(x)` and the conditional `LL(x | P)`. The **ReCaLL score** is

```
ReCaLL(x) = LL(x | P) / LL(x)
```

Because log-likelihoods are negative and conditioning on the non-member prefix lowers the LL *more* for members (`LL(x_m | P) < LL(x_m)`), members receive scores typically greater than 1 and higher than non-members: `E[ReCaLL(x_member)] > E[ReCaLL(x_non-member)]`. Threshold the score; **higher => member**. The paper reports member avg ReCaLL ~ 1.20 vs non-member ~ 1.04 (Figure 2, Pythia-6.9B, WikiMIA-32, 5-shot).

**Benchmarks / models / baselines / metrics.** Evaluated on **WikiMIA** (length splits 32 / 64 / 128) and **MIMIR** (Pile, with 7-gram and 13-gram filtering). WikiMIA models: Pythia-6.9B, GPT-NeoX-20B, LLaMA-30B, OPT-66B, Mamba; MIMIR models: the Pythia family (160M-12B). Synthetic prefixes are generated by GPT-4o. Baselines: LOSS, Reference, Zlib, Neighbor, Min-K%, Min-K%++. Metrics: **AUC** (main) and **TPR@1%FPR**. ReCaLL is SOTA on WikiMIA, beating runner-up Min-K%++ by ~14.8-15.4% AUC across lengths, and is competitive on MIMIR. The only hyperparameter is `n` (shots); even a **single shot** beats all baselines, and a **fixed** prefix beats per-target dynamic prefixes.


## Baseline Attack Definition

**Target record.** A candidate text sequence `x`. Members are sequences present in the target model's training (or fine-tuning) set; non-members are distribution-matched held-out sequences.

**Two log-likelihoods per candidate.** From the target model `M` compute:

- `LL(x)` -- the unconditional total log-likelihood of the sequence (a negative number).
- `LL(x | P)` -- the total log-likelihood of the sequence when the *fixed* non-member prefix `P` is prepended (only the `x` tokens are scored).

**Score.** `ReCaLL(x) = LL(x | P) / LL(x)`. Since both terms are negative and conditioning depresses the member's LL more, the ratio is **higher** for members. This matches the `>=` threshold convention used across these recreations (higher membership score => more likely member), so no sign flip is needed -- the ratio is already oriented so members score higher.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Sequence

SOURCE_SUMMARY = Path("../../papers/summary/08_recall.md")
ATTACK_NAME = "recall"


def recall_score(ll_x: float, ll_x_given_prefix: float) -> float:
    """ReCaLL membership score = LL(x | P) / LL(x).

    Both arguments are (negative) total log-likelihoods under the target model:
    the unconditional LL(x) and the prefix-conditioned LL(x | P). Conditioning on
    a non-member prefix lowers the LL more for members, so the ratio is larger
    (typically > 1) for members and closer to 1 for non-members. Higher => member.
    """
    if ll_x == 0.0:
        raise ValueError("LL(x) must be non-zero to form the ReCaLL ratio.")
    return ll_x_given_prefix / ll_x


@dataclass(frozen=True)
class CandidateScore:
    text: str
    truth_member: bool
    ll_x: float                # unconditional total log-likelihood LL(x) (negative)
    ll_x_given_prefix: float   # conditional total log-likelihood LL(x | P) (negative)

    @property
    def membership_score(self) -> float:
        # ReCaLL ratio, already oriented so members score higher.
        return recall_score(self.ll_x, self.ll_x_given_prefix)

    @property
    def ll_drop(self) -> float:
        # How much conditioning lowered the LL (positive => LL dropped). Larger for members.
        return self.ll_x - self.ll_x_given_prefix

## Optional Hugging Face Scoring

Use this cell for a real target model such as `EleutherAI/pythia-6.9b` (a paper target) or any fine-tuned checkpoint. ReCaLL needs **no second model** -- only two forward passes of the same target model per candidate (one unconditional, one with the fixed non-member prefix prepended). The smoke test below does not require these packages or any model download.


In [ ]:
def sequence_loglik_hf(model, tokenizer, text: str, prefix: str = None, device: str = "cpu", max_length: int = 1024) -> float:
    """Total log-likelihood of `text`, optionally conditioned on a fixed `prefix`.

    Returns a NEGATIVE float: the sum of per-token log-probabilities over the
    `text` tokens. When `prefix` is provided, the prefix tokens are prepended but
    excluded from the loss (labels = -100), so the returned value is the
    conditional log-likelihood LL(text | prefix). With prefix=None it is the
    unconditional LL(text). The ReCaLL ratio is
        recall_score(LL(text), LL(text | prefix)).
    """
    import torch

    if prefix:
        prefix_ids = tokenizer(prefix, return_tensors="pt").input_ids
        text_ids = tokenizer(text, return_tensors="pt").input_ids
        input_ids = torch.cat([prefix_ids, text_ids], dim=1)[:, :max_length]
        labels = input_ids.clone()
        labels[:, : prefix_ids.shape[1]] = -100  # score only the text tokens
    else:
        input_ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).input_ids
        labels = input_ids.clone()

    input_ids = input_ids.to(device)
    labels = labels.to(device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=labels)

    # Hugging Face returns the MEAN NLL over scored (label != -100, post-shift) tokens.
    shift_labels = labels[:, 1:]
    num_scored = int((shift_labels != -100).sum().item())
    if num_scored == 0:
        raise ValueError("Need at least one scored text token.")
    total_nll = float(outputs.loss.detach().cpu()) * num_scored
    return -total_nll  # log-likelihood is negative


def score_texts_with_hf(target_model, tokenizer, texts: Sequence[str], labels: Sequence[bool], prefix: str, device: str = "cpu", max_length: int = 1024) -> List[CandidateScore]:
    """Score candidates with a real target model using a single FIXED non-member prefix."""
    rows = []
    for text, truth_member in zip(texts, labels):
        ll_x = sequence_loglik_hf(target_model, tokenizer, text, prefix=None, device=device, max_length=max_length)
        ll_xp = sequence_loglik_hf(target_model, tokenizer, text, prefix=prefix, device=device, max_length=max_length)
        rows.append(CandidateScore(text=text, truth_member=bool(truth_member), ll_x=ll_x, ll_x_given_prefix=ll_xp))
    return rows

## Thresholding and Metrics

The paper's headline metric is threshold-free **AUC** (plus TPR@1%FPR). For small controlled trials this notebook additionally reports thresholded confusion counts, TPR, TNR, attack advantage, accuracy, precision, recall, and F1, plus a rank-based ROC-AUC identical in spirit to the paper's main metric.


In [ ]:
def predict_membership(rows: Sequence[CandidateScore], threshold: float) -> List[bool]:
    return [row.membership_score >= threshold for row in rows]


def confusion_counts(labels: Sequence[bool], preds: Sequence[bool]):
    tp = sum(1 for y, p in zip(labels, preds) if y and p)
    tn = sum(1 for y, p in zip(labels, preds) if not y and not p)
    fp = sum(1 for y, p in zip(labels, preds) if not y and p)
    fn = sum(1 for y, p in zip(labels, preds) if y and not p)
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn}


def roc_auc(labels: Sequence[bool], scores: Sequence[float]) -> float:
    """Rank-based ROC-AUC (probability a random member outranks a random non-member)."""
    pos = [s for y, s in zip(labels, scores) if y]
    neg = [s for y, s in zip(labels, scores) if not y]
    if not pos or not neg:
        return float("nan")
    wins = 0.0
    for p in pos:
        for n in neg:
            wins += 1.0 if p > n else (0.5 if p == n else 0.0)
    return wins / (len(pos) * len(neg))


def metric_summary(rows: Sequence[CandidateScore], preds: Sequence[bool]):
    labels = [row.truth_member for row in rows]
    counts = confusion_counts(labels, preds)
    tp, tn, fp, fn = counts["tp"], counts["tn"], counts["fp"], counts["fn"]
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tpr
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        **counts,
        "tpr": tpr,
        "tnr": tnr,
        "adv": 0.5 * tpr + 0.5 * tnr,
        "accuracy": (tp + tn) / len(labels) if labels else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc(labels, [row.membership_score for row in rows]),
    }


def percentile_threshold(rows: Sequence[CandidateScore], member_fraction: float = 0.5) -> float:
    scores = sorted(row.membership_score for row in rows)
    if not scores:
        raise ValueError("Cannot threshold an empty score list.")
    index = max(0, min(len(scores) - 1, int((1.0 - member_fraction) * len(scores))))
    return scores[index]

## Synthetic Smoke Recreation

The synthetic table emulates the expected ReCaLL signal using directly-constructed log-likelihoods (no model download):

- **Members** are records the model has memorized. Conditioning on the fixed non-member prefix `P` perturbs their already-memorized predictive distribution strongly, so `LL(x | P)` drops well below `LL(x)` -- a large negative shift -- and the ratio `LL(x|P)/LL(x)` climbs **above** the non-members'.
- **Non-members** are unfamiliar to the model, so the prefix barely changes their LL: `LL(x | P) ~ LL(x)`, keeping the ratio close to 1.

The numbers are chosen so every member outranks every non-member, exactly as the paper's Figure 2 distributions show (member avg ~1.20 vs non-member avg ~1.04). This is a runnable correctness check that the `recall_score` and the metrics pipeline behave as the paper describes -- not a substitute for the full WikiMIA / MIMIR experiment.


In [ ]:
def synthetic_recall_scores() -> List[CandidateScore]:
    return [
        # Members: memorized. Prefix depresses LL a lot -> ratio well above 1.
        CandidateScore("Patient Ana Ortiz, MRN 84213, was prescribed 12 units of insulin nightly.", True, ll_x=-5.00, ll_x_given_prefix=-6.55),   # ratio 1.31
        CandidateScore("API_SECRET_KEY = sk-live-9f3a2b7c4d8e1f6a0c5b2d9e7f4a1c3b", True, ll_x=-4.00, ll_x_given_prefix=-5.12),                     # ratio 1.28
        # Non-members: unfamiliar. Prefix barely changes LL -> ratio near 1.
        CandidateScore("The committee will reconvene next quarter to review the proposal.", False, ll_x=-6.00, ll_x_given_prefix=-6.18),           # ratio 1.03
        CandidateScore("Weather permitting, the community picnic will be held on Saturday afternoon.", False, ll_x=-5.50, ll_x_given_prefix=-5.61),  # ratio 1.02
    ]


def run_recreation_smoke_test():
    rows = synthetic_recall_scores()
    threshold = percentile_threshold(rows, member_fraction=0.5)
    preds = predict_membership(rows, threshold=threshold)
    metrics = metric_summary(rows, preds)

    # The two memorized records must rank above both held-out records.
    assert metrics["tp"] == 2, metrics
    assert metrics["tn"] == 2, metrics
    assert metrics["adv"] == 1.0, metrics
    assert metrics["roc_auc"] == 1.0, metrics

    # ReCaLL sanity: conditioning must drop the members' LL more than the
    # non-members', and members must all score above 1 (Figure 2 behaviour).
    members = [r for r in rows if r.truth_member]
    non_members = [r for r in rows if not r.truth_member]
    assert min(m.membership_score for m in members) > max(n.membership_score for n in non_members), \
        "ReCaLL ratio failed to rank members above non-members"
    assert min(m.ll_drop for m in members) > max(n.ll_drop for n in non_members), \
        "members should see a larger conditional LL drop than non-members"
    assert all(m.membership_score > 1.0 for m in members), "members should have ReCaLL > 1"

    return {
        "threshold": threshold,
        "metrics": metrics,
        "ranking": [
            {"text": r.text[:40], "member": r.truth_member,
             "ll_x": round(r.ll_x, 3),
             "ll_x_given_prefix": round(r.ll_x_given_prefix, 3),
             "recall_score": round(r.membership_score, 6)}
            for r in sorted(rows, key=lambda r: r.membership_score, reverse=True)
        ],
    }

smoke_result = run_recreation_smoke_test()
smoke_result

## How to Run a Real Recreation

1. Load the target language model with `AutoModelForCausalLM` (the paper uses e.g. `EleutherAI/pythia-6.9b`; any fine-tuned checkpoint works). No reference model is needed.
2. Build a **fixed** non-member prefix `P` by concatenating `n` shots known to be non-members -- recent post-cutoff text, user-generated text, or GPT-4o-synthesized text. The paper sweeps `n` from 1 to 12 and reports the best; even one shot beats all baselines, and a fixed prefix beats per-target dynamic prefixes.
3. Collect matched member / non-member candidate texts (WikiMIA length splits 32 / 64 / 128, or MIMIR).
4. Call `score_texts_with_hf(target_model, tokenizer, texts, labels, prefix=P)` -- it computes `LL(x)` and `LL(x | P)` with `sequence_loglik_hf` and forms `ReCaLL(x) = LL(x|P)/LL(x)` per candidate.
5. Rank by `membership_score` and report `roc_auc` (the paper's main AUC metric) via `metric_summary`, or threshold with `predict_membership`. Optionally use the **ensemble** variant: split many shots into smaller prefix sets, compute a ReCaLL score per set, and average to reduce variance and sidestep the context-window limit.

For the federated-learning fine-tuning adaptation of this attack, see `../adaptations/recall_adaptations.ipynb`.
